
# 05_Neural_LV_CPU_final_edited

## Final CPU Neural Local Volatility baseline

This is the canonical CPU implementation for the UROP benchmark. It runs **top-to-bottom from the four raw March-2024 SPX OptionMetrics CSVs**, creates its own cache, and does not require a checkpoint from an older notebook.

### Model and development history

The risk-neutral forward-normalised state is
$$
X_t=\frac{S_t}{F_0(t)}, \qquad
dX_t=X_t\,\sigma_\theta(t,X_t)\,dW_t ,
$$
with a neural local-volatility coefficient and the same tamed-Euler discretisation used throughout v5–v7. A learned stochastic-integral **auxiliary control variate**
$$
\Phi^{cv}=\Phi-\int_0^T h_\xi(t,X_t)\,dW_t
$$
is trained alongside the NLV to reduce Monte Carlo variance. It is distinct from the finite-difference NLV delta reported in the hedging experiment.

The strong 1-March model used by the successful rolling experiments had a genuine inter-version calibration lineage, reproduced here rather than replaced by a short random cold start:

1. **v5 initial calibration:** shared 4×50 ReLU NLV, 1,000 alternating NLV/control-variate epochs on the capped OTM surface;
2. **v6 architecture transfer under the final-v7 objective:** the trained shared trunk/output are copied into the raw-loss **maturity-head** 4×50 architecture, then trained for 600 alternating epochs on a deterministic 80% split of the **full filtered OTM surface**, using uniform contract sampling, contract-level raw-price MSE, a constant $10^{-4}$ NLV learning rate and $10^{-3}$ control-variate learning rate;
3. **v7-final full-sample refinement:** the inherited maturity-head model is refined on **all** filtered 1-March OTM observations, with update 0 **guaranteed to remain in the final 50k rerank candidate set**.

After that one-time initial calibration, the selected CPU method is deliberately simple: each later date is warm-started from the previous fitted surface and receives **exactly 15 NLV updates**, with no early stopping and no same-day checkpoint reranking.

### Frozen CPU specification

- NLV: shared 4×50 ReLU trunk + maturity-specific Softplus heads;
- auxiliary control variate: interval bank of 3×30 ReLU networks;
- tamed Euler: 96 steps/year, with quoted maturities inserted exactly;
- rolling objective: contract-level raw-price MSE, spot-scaled only for numerical conditioning;
- rolling batch: 4,096 paths, 400 contracts, 10 NLV + 10 control-variate optimiser steps per update;
- Adam learning rates: $10^{-4}$ for NLV and $10^{-3}$ for the control variate;
- primary next-day universe: matched contracts OTM on both dates;
- model delta: same-contract finite difference with forward-consistent perturbation and a 5 bp headline bump;
- one-day hedging: self-financing delta-hedge error including cash financing and dividend carry, with the old $\Delta C-\Delta\,\Delta S$ residual retained only as a diagnostic;
- primary evaluation: 100k paths; final evaluation: 400k antithetic paths.

### Temporal protocol

- **Development:** 1 March → 8 March 2024.
- **Untouched out-of-sample test:** 8 March → 28 March 2024 (14 next-trading-day transitions).

The test period is not used to change the model. The GPU notebook will later test the remaining Sabaté/Gierjatowicz fidelity gaps; they are intentionally not mixed into this CPU benchmark.


In [1]:

from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
import copy, faulthandler, gc, hashlib, json, math, os, platform, random, sys, traceback, warnings

# Conservative CPU execution retained from the crash-safe v7 notebook.
for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "2"

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq
from scipy.special import ndtr
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
INITIAL_DATE = pd.Timestamp("2024-03-01")
DEVELOPMENT_END = pd.Timestamp("2024-03-08")
TEST_END = pd.Timestamp("2024-03-28")
DEVICE = torch.device("cpu")
torch.set_num_threads(2)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

def set_seed(seed):
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))

set_seed(SEED)

# Frozen architecture and SDE discretisation.
SDE_HIDDEN_LAYERS, SDE_WIDTH = 4, 50
HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH = 3, 30
BASE_STEPS_PER_YEAR = 96

# Historical initial-calibration lineage (v5 -> v6 -> v7 full-sample refinement).
LEGACY_LR = 1e-3
LEGACY_PATHS = 4_096
LEGACY_STEPS_PER_EPOCH = 10
LEGACY_QUOTES_PER_MATURITY = 12
LEGACY_MONITOR_PATHS = 16_384
LEGACY_EVAL_EVERY_SDE = 10
LEGACY_TOP_K = 5
LEGACY_RERANK_PATHS = 50_000
LEGACY_SAVE_EVERY = 10
V5_EPOCHS = 1_000
V6_HEAD_EPOCHS = 600

# Evaluation.
PRIMARY_EVAL_PATHS = 100_000
FINAL_EVAL_PATHS = 400_000
FINAL_MC_PATHS = PRIMARY_EVAL_PATHS  # compatibility with audited v7 evaluation functions
FINAL_PATH_CHUNK = 2_000
DELTA_BUMP_BPS = (1, 5, 10)
DELTA_HEADLINE_BPS = 5

CACHE_DIR = Path("neural_lv_cpu_final_edited_cache")
INITIAL_DIR = CACHE_DIR / "initial_calibration"
ROLLING_DIR = CACHE_DIR / "rolling_models"
RESULT_DIR = CACHE_DIR / "results"
PLOT_DIR = CACHE_DIR / "plots"
for folder in (CACHE_DIR, INITIAL_DIR, ROLLING_DIR, RESULT_DIR, PLOT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def cpu_state(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

def atomic_save(payload, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp); os.replace(tmp, path)

# Native-fault diagnostics stay enabled for the whole kernel session.
if "_fault" in globals():
    faulthandler.disable(); _fault.close()
try:
    import psutil
except ImportError:
    psutil = None

_fault = open(CACHE_DIR / "native_fault.log", "a", buffering=1)
faulthandler.enable(file=_fault, all_threads=True)

def stage_log(message):
    rss = f"{psutil.Process().memory_info().rss/2**20:.0f} MiB" if psutil else "n/a"
    line = f"{datetime.now().isoformat(timespec='seconds')} | {message} | RSS={rss}"
    print(line, flush=True)
    with open(CACHE_DIR / "stage_trace.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")

(CACHE_DIR / "environment.json").write_text(json.dumps({
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "device": str(DEVICE), "torch_threads": torch.get_num_threads(),
}, indent=2), encoding="utf-8")
stage_log("Session initialised")


2026-09-24T16:31:40 | Session initialised | RSS=318 MiB



## 1. Data, market curves and calibration universes

The CSVs are read directly from the notebook directory. The run records their exact filenames, sizes and SHA-256 hashes.

The final v7 filters are retained: European SPX contracts, 7–365 DTE, valid positive quotes, bounded relative spread, reasonable market IV/delta, static price bounds, and $|\log(K/F)|\le0.30$. Calls and puts are collapsed to one OTM observation per strike; near the forward, the tighter-spread side is used.

Two OTM universes are retained for the staged initialization:

- **Stage I (historical v5 pretraining):** at most 25 observations per expiry/settlement slice;
- **Stage II onward:** every filtered OTM observation. Stage II takes a deterministic 80/20 split of this full surface; Stage III and all rolling dates use the full filtered OTM universe.


In [2]:

DATA_DIR = Path.cwd()
DATA_FILES = {
    "options": DATA_DIR / "one_month_spx_option_price.csv",
    "forwards": DATA_DIR / "one_month_spx_forward_price.csv",
    "spot": DATA_DIR / "one_month_spx_security_price.csv",
    "rates": DATA_DIR / "one_month_spx_zero_coupon_yield_curve.csv",
}
missing = [p.name for p in DATA_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Place the four SPX CSV files beside this notebook. Missing: " + ", ".join(missing)
    )

options_raw = pd.read_csv(DATA_FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(DATA_FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(DATA_FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(DATA_FILES["rates"], low_memory=False)

def file_sha256(path):
    with open(path, "rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

manifest = {
    k: {"file": p.name, "bytes": p.stat().st_size, "sha256": file_sha256(p)}
    for k, p in DATA_FILES.items()
}
(CACHE_DIR / "data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(pd.DataFrame(manifest).T[["file", "bytes"]])


SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8


def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]
    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6) & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)


def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND
    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm", "optionid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )


quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM calibration observations:", len(otm))
print("Trading dates:", otm["date"].nunique())
print("Expiry/settlement slices:", otm.groupby(["date", "exdate", "am_settlement"]).ngroups)


                                               file     bytes
options              one_month_spx_option_price.csv  54286695
forwards            one_month_spx_forward_price.csv     55900
spot               one_month_spx_security_price.csv      1221
rates     one_month_spx_zero_coupon_yield_curve.csv      5193
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809


In [3]:
@dataclass
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(self.T_nodes, self.logD_nodes, extrapolate=True)
        self._logF = PchipInterpolator(self.T_nodes, self.logF_nodes, extrapolate=True)

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))


class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
        spot["date"] = pd.to_datetime(spot["date"])
        self.spot = spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(self.rates["rate"], errors="coerce") / 100.0
        self._cache = {}

    def get(self, date, am_settlement):
        date, am = pd.Timestamp(date), int(am_settlement)
        key = (date, am)
        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])
        rg = self.rates.loc[self.rates["date"] == date, ["days", "rate_decimal"]].dropna().sort_values("days")
        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date) & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()
        if len(fg) < 2:
            fg = self.forwards.loc[self.forwards["date"] == date, ["expiration", "ForwardPrice"]].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"].mean().sort_values("T")
        )
        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)
        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(date, am, S0, T_all, logD_all, logF_all)
        self._cache[key] = curve
        return curve

curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)

def _rename_selected(df):
    out = df.rename(columns={
        "mid_otm": "target_mid", "cp_otm": "target_cp", "optionid_otm": "optionid"
    }).reset_index(drop=True)
    out["full_selected_id"] = np.arange(len(out), dtype=int)
    return out

def capped_otm_universe(date, cap=25):
    """Historical Stage-I v5 25-quotes-per-expiry/settlement surface."""
    day = otm.loc[otm["date"] == pd.Timestamp(date)].copy()
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        pos = np.arange(len(g)) if len(g) <= cap else np.unique(np.round(np.linspace(0, len(g)-1, cap)).astype(int))
        pieces.append(g.iloc[pos])
    return _rename_selected(pd.concat(pieces, ignore_index=True))

def all_otm_universe(date):
    """Final-v7 calibration universe: every filtered OTM observation."""
    return _rename_selected(otm.loc[otm["date"] == pd.Timestamp(date)].copy())

def stratified_holdout(selected, fraction=0.20, seed=SEED+77):
    """Deterministic ~80/20 split within each expiry/settlement slice."""
    selected = selected.copy().reset_index(drop=True)
    hold = np.zeros(len(selected), dtype=bool)
    rng = np.random.default_rng(seed)
    stride = max(2, int(round(1.0/fraction)))
    for _, g in selected.groupby(["exdate", "am_settlement"], sort=False):
        ordered = g.sort_values("k").index.to_numpy()
        chosen = ordered[int(rng.integers(0, stride))::stride]
        if len(chosen) == 0: chosen = ordered[[len(ordered)//2]]
        if len(chosen) >= len(ordered): chosen = chosen[:-1]
        hold[chosen] = True
    train = selected.loc[~hold].copy().reset_index(drop=True)
    holdout = selected.loc[hold].copy().reset_index(drop=True)
    train["contract_id"] = np.arange(len(train), dtype=int)
    return train, holdout

trading_dates = sorted(pd.to_datetime(quotes_all["date"].unique()))
otm_option_ids_by_date = {
    d: set(otm.loc[otm["date"] == d, "optionid_otm"].astype(int))
    for d in trading_dates
}

initial_capped = capped_otm_universe(INITIAL_DATE)
initial_all_otm = all_otm_universe(INITIAL_DATE)
initial_train80, initial_holdout20 = stratified_holdout(initial_all_otm)
print("1-Mar capped OTM (Stage I only):", len(initial_capped),
      "| full filtered OTM:", len(initial_all_otm),
      "| Stage-II full-OTM train/holdout:", len(initial_train80), len(initial_holdout20))


1-Mar capped OTM (Stage I only): 997 | full filtered OTM: 4304 | Stage-II full-OTM train/holdout: 3443 861



## 2. Neural networks, SDE simulation and rolling time transfer

The maturity-head NLV retains a shared representation of $(t,X)$ but has one positive Softplus output head per maturity interval. For rolling recalibration, the first-layer time coefficient is shifted exactly so that moving the valuation origin forward does not change the learned calendar-time function; heads are matched by nearest calendar expiry.

The same simulation core supports the historical maturity-balanced **Stage-I v5 pretraining** and the uniform-contract Stage-II/Stage-III/rolling calibration used by the final v7 specification.


In [4]:
class FeedForwardTrunk(nn.Module):
    def __init__(self, input_dim, hidden_layers, width):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(hidden_layers):
            layers += [nn.Linear(d, width), nn.ReLU()]
            d = width
        self.net = nn.Sequential(*layers)
        self.output_dim = width

    def forward(self, x):
        return self.net(x)


def sabate_weight_initialisation(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)


def _tx(t, x):
    if x.ndim == 1:
        x = x[:, None]
    if not torch.is_tensor(t):
        t = torch.full_like(x, float(t))
    elif t.ndim == 0:
        t = torch.full_like(x, float(t.item()))
    elif t.ndim == 1:
        t = t[:, None]
    return torch.cat([t, x], dim=1)


class SharedNeuralLocalVol(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, SDE_HIDDEN_LAYERS, SDE_WIDTH)
        self.out = nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
        self.apply(sabate_weight_initialisation)

    def forward(self, t, x):
        return self.out(self.trunk(_tx(t, x)))


class MaturityHeadNeuralLocalVol(nn.Module):
    def __init__(self, maturity_boundaries):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.trunk = FeedForwardTrunk(2, SDE_HIDDEN_LAYERS, SDE_WIDTH)
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
            for _ in self.boundaries
        ])
        self.apply(sabate_weight_initialisation)

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.heads)-1)

    def forward(self, t, x):
        h = self.trunk(_tx(t, x))
        return self.heads[self.interval_index(t)](h)


class PerOptionHedgeNet(nn.Module):
    def __init__(self, n_options):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH)
        self.output = nn.Linear(HEDGE_WIDTH, int(n_options))
        self.apply(sabate_weight_initialisation)

    def forward_ids(self, t, x, option_ids):
        h = self.trunk(_tx(t, x))
        ids = torch.as_tensor(option_ids, dtype=torch.long, device=h.device)
        return h @ self.output.weight[ids].T + self.output.bias[ids][None, :]


class IntervalHedgeBank(nn.Module):
    def __init__(self, maturity_boundaries, n_options):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.nets = nn.ModuleList([PerOptionHedgeNet(n_options) for _ in self.boundaries])

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.nets)-1)

    def forward_ids(self, t, x, option_ids):
        return self.nets[self.interval_index(t)].forward_ids(t, x, option_ids)


def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(value)


def safe_torch_load(path, map_location=DEVICE):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def transfer_rolling_model(previous, current, previous_date, current_date):
    """Warm-start by calendar time, including the trunk's explicit time input.

    The shared trunk receives [t, x]. Advancing the valuation date by elapsed means that
    the old function at the same calendar point is h_old(t + elapsed, x). Because the
    first trunk layer is affine, this shift is represented exactly by
        b_new = b_old + W_time * elapsed.
    Heads are mapped to the old boundary nearest T_new + elapsed.
    """
    elapsed = (pd.Timestamp(current_date) - pd.Timestamp(previous_date)).days / 365.0
    if elapsed < -1e-12:
        raise ValueError("Rolling transfer requires a non-decreasing calendar date.")

    current.trunk.load_state_dict(previous.trunk.state_dict())

    # Exact time-input shift in the first affine layer.
    first = current.trunk.net[0]
    if not isinstance(first, nn.Linear) or first.weight.shape[1] != 2:
        raise RuntimeError("Unexpected trunk first layer; cannot apply exact time shift.")
    if abs(elapsed) > 0:
        with torch.no_grad():
            first.bias.add_(first.weight[:, 0] * float(elapsed))

    matched = []
    with torch.no_grad():
        for i, T_new in enumerate(current.boundaries):
            target_old_T = float(T_new) + elapsed
            j = int(np.argmin(np.abs(previous.boundaries - target_old_T)))
            current.heads[i].load_state_dict(previous.heads[j].state_dict())
            matched.append(abs(float(previous.boundaries[j]) - target_old_T))

    print(
        f"Warm-start {pd.Timestamp(previous_date).date()} -> {pd.Timestamp(current_date).date()}: "
        f"{len(current.heads)} heads | elapsed={elapsed:.5f}y | "
        f"median head mismatch={np.median(matched):.5f}y"
    )


@torch.no_grad()
def trunk_time_shift_unit_test():
    prev = MaturityHeadNeuralLocalVol([0.10, 0.25, 0.50]).to(DEVICE).eval()
    cur = MaturityHeadNeuralLocalVol([0.09, 0.24, 0.49]).to(DEVICE).eval()
    d0, d1 = pd.Timestamp("2024-03-01"), pd.Timestamp("2024-03-04")
    transfer_rolling_model(prev, cur, d0, d1)
    elapsed = (d1-d0).days/365.0

    x = torch.linspace(0.8, 1.2, 31, device=DEVICE)[:, None]
    t = torch.full_like(x, 0.13)
    a = prev.trunk(_tx(t + elapsed, x))
    b = cur.trunk(_tx(t, x))
    err = float(torch.max(torch.abs(a-b)).cpu())
    if err > 2e-6:
        raise RuntimeError(f"Trunk time-shift transfer failed: max error {err:.3e}")
    print(f"Trunk time-shift unit test max error={err:.2e}")
    return err

def initialise_maturity_heads_from_shared(shared, headed):
    """v6 transfer: copy the shared trunk and shared Softplus output into every maturity head."""
    headed.trunk.load_state_dict(shared.trunk.state_dict(), strict=True)
    with torch.no_grad():
        for head in headed.heads:
            head[0].weight.copy_(shared.out[0].weight)
            head[0].bias.copy_(shared.out[0].bias)
    return headed


In [5]:
def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.sort(np.unique(np.asarray(maturities, float)))
    max_T = float(maturities[-1])
    base = np.arange(0.0, max_T + 0.5/steps_per_year, 1.0/steps_per_year)
    base = base[base <= max_T + 1e-12]
    grid = np.unique(np.r_[0.0, base, maturities, max_T])
    grid.sort()
    return grid


def nearest_grid_indices(grid, times):
    grid, times = np.asarray(grid), np.asarray(times)
    idx = np.clip(np.searchsorted(grid, times), 0, len(grid)-1)
    left = np.maximum(idx-1, 0)
    idx = np.where(np.abs(grid[left]-times) < np.abs(grid[idx]-times), left, idx)
    if np.max(np.abs(grid[idx]-times)) > 1e-10:
        raise RuntimeError("A listed maturity was not represented exactly on the simulation grid.")
    return idx


@dataclass
class CalibrationContext:
    date: pd.Timestamp
    spot: float
    contracts: pd.DataFrame
    maturity_boundaries: np.ndarray
    grid_np: np.ndarray
    grid: torch.Tensor
    maturity_index: torch.Tensor
    stage_index: np.ndarray
    stage_tensor: torch.Tensor
    K: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor
    F_grid: torch.Tensor
    D_grid: torch.Tensor


def build_calibration_context(date, contracts):
    date = pd.Timestamp(date)
    contracts = contracts.copy().reset_index(drop=True)
    contracts["contract_id"] = np.arange(len(contracts), dtype=int)
    boundaries = np.sort(contracts["T"].unique())
    grid_np = build_time_grid(boundaries)
    maturity_idx = nearest_grid_indices(grid_np, contracts["T"].to_numpy(float))
    stage_map = {round(float(T),12): i for i,T in enumerate(boundaries)}
    stage_index = np.array([stage_map[round(float(T),12)] for T in contracts["T"]], dtype=int)

    S0 = float(contracts["S"].iloc[0])
    F_grid = np.empty((2, len(grid_np)))
    D_grid = np.empty((2, len(grid_np)))
    for am in [0,1]:
        curve = curve_book.get(date, am)
        F_grid[am] = curve.forward(grid_np)
        D_grid[am] = curve.discount(0.0, grid_np)

    return CalibrationContext(
        date=date, spot=S0, contracts=contracts, maturity_boundaries=boundaries,
        grid_np=grid_np, grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        maturity_index=torch.tensor(maturity_idx, dtype=torch.long, device=DEVICE),
        stage_index=stage_index,
        stage_tensor=torch.tensor(stage_index, dtype=torch.long, device=DEVICE),
        K=torch.tensor(contracts["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(contracts["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(contracts["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(contracts["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(np.where(contracts["target_cp"].eq("C"),1.0,-1.0), dtype=torch.float32, device=DEVICE),
        am=torch.tensor(contracts["am_settlement"].to_numpy(int), dtype=torch.long, device=DEVICE),
        F_grid=torch.tensor(F_grid, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid, dtype=torch.float32, device=DEVICE),
    )


def antithetic_normals(total_paths, n_steps, seed):
    half = (total_paths+1)//2
    gen = torch.Generator(device="cpu"); gen.manual_seed(int(seed))
    z = torch.randn(half, n_steps, generator=gen, dtype=torch.float32)
    return torch.cat([z,-z], dim=0)[:total_paths].to(DEVICE)


def empirical_payoff_mean(samples, strike_norm, cp_flag):
    x = np.sort(np.asarray(samples,float)); n = len(x)
    prefix = np.r_[0.0, np.cumsum(x)]; total = prefix[-1]
    k = np.asarray(strike_norm,float); idx = np.searchsorted(x,k,side="right")
    below, above = prefix[idx], total-prefix[idx]
    call = (above-k*(n-idx))/n
    put = (k*idx-below)/n
    return np.where(np.asarray(cp_flag)=="C", call, put)


def maturity_balanced_mean(values, stages):
    return torch.stack([values[stages == s].mean() for s in torch.unique(stages)]).mean()


def simulate_estimators(model, hedge, ctx, contract_ids, z, phase):
    ids_np = np.asarray(contract_ids,dtype=int)
    ids = torch.as_tensor(ids_np,dtype=torch.long,device=DEVICE)
    maturity_idx = ctx.maturity_index[ids]
    K, F_T, D_T = ctx.K[ids], ctx.F_T[ids], ctx.D_T[ids]
    cp, am = ctx.cp_sign[ids], ctx.am[ids]

    B, M = z.shape[0], len(ids_np)
    x = torch.ones((B,1),dtype=torch.float32,device=DEVICE)
    cv = torch.zeros((B,M),dtype=torch.float32,device=DEVICE)
    raw = torch.zeros((B,M),dtype=torch.float32,device=DEVICE)

    for step in range(int(maturity_idx.max().item())):
        t = ctx.grid[step]; dt = ctx.grid[step+1]-ctx.grid[step]; sqrt_dt = torch.sqrt(dt)
        sigma = model(t,x) if phase == "sde" else model(t,x).detach()
        dW = sqrt_dt*z[:,step:step+1]

        active = torch.where(maturity_idx > step)[0]
        if len(active):
            active_ids = ids_np[active.detach().cpu().numpy()]
            h = hedge.forward_ids(t,x.detach(),active_ids)
            if phase == "sde":
                h = h.detach()
            stochastic_piece = x.detach()*sigma.detach()*dW
            discounted_forward = ctx.D_grid[am[active],step]*ctx.F_grid[am[active],step]
            cv[:,active] += h*stochastic_piece*discounted_forward[None,:]

        a = x*sigma
        x = x + a*dW/(1.0 + torch.abs(x.detach()*sigma.detach())*sqrt_dt)

        matured = torch.where(maturity_idx == step+1)[0]
        if len(matured):
            S_T = x*F_T[matured][None,:]; strike = K[matured][None,:]
            call = torch.clamp(S_T-strike,min=0.0); put = torch.clamp(strike-S_T,min=0.0)
            raw[:,matured] = torch.where((cp[matured]>0)[None,:],call,put)*D_T[matured][None,:]

    return raw, raw-cv


class MaturityBalancedSampler:
    def __init__(self, ctx, per_maturity, seed):
        self.groups = [np.where(ctx.stage_index==j)[0] for j in range(len(ctx.maturity_boundaries))]
        self.per_maturity = per_maturity
        self.rng = np.random.default_rng(seed)

    def draw(self):
        out = []
        for g in self.groups:
            n = min(self.per_maturity, len(g))
            out.extend(self.rng.choice(g,size=n,replace=False))
        return np.asarray(out,dtype=int)


@torch.no_grad()
def simulate_terminal_x(model, ctx, z):
    x = torch.ones((z.shape[0],1),dtype=torch.float32,device=DEVICE)
    stage_by_grid = {int(nearest_grid_indices(ctx.grid_np,[T])[0]): j for j,T in enumerate(ctx.maturity_boundaries)}
    terminal = {}
    for step in range(len(ctx.grid_np)-1):
        t = ctx.grid[step]; dt = ctx.grid[step+1]-ctx.grid[step]; sqrt_dt = torch.sqrt(dt)
        sigma = model(t,x); dW = sqrt_dt*z[:,step:step+1]
        a = x*sigma; x = x + a*dW/(1.0+torch.abs(a)*sqrt_dt)
        if step+1 in stage_by_grid:
            terminal[stage_by_grid[step+1]] = x.squeeze(1).detach().cpu().numpy().copy()
    return terminal


def selected_prices_from_terminal(ctx, terminal):
    rows = []
    for stage in range(len(ctx.maturity_boundaries)):
        ids = np.where(ctx.stage_index==stage)[0]
        g = ctx.contracts.iloc[ids]
        payoff = empirical_payoff_mean(
            terminal[stage], g["K"].to_numpy(float)/g["F"].to_numpy(float), g["target_cp"].to_numpy()
        )
        price = g["D"].to_numpy(float)*g["F"].to_numpy(float)*payoff
        tmp = g[["contract_id","optionid","T","k","target_cp","target_mid"]].copy()
        tmp["raw_model_price"] = price
        tmp["raw_error"] = price-tmp["target_mid"].to_numpy(float)
        rows.append(tmp)
    return pd.concat(rows,ignore_index=True)


def objective_diagnostics(model, ctx, z_fixed):
    """Final-v7 diagnostics: every eligible contract has equal weight."""
    table = selected_prices_from_terminal(ctx, simulate_terminal_x(model,ctx,z_fixed))
    e = table["raw_error"].to_numpy(float)
    return {
        "raw_rmse": float(np.sqrt(np.mean(e**2))),
        "raw_mae": float(np.mean(np.abs(e))),
        "table": table,
    }


class UniformContractSampler:
    """Uniform sampling from the empirical training contracts."""
    def __init__(self, ctx, batch_size, seed):
        self.n_total = len(ctx.contracts)
        self.batch_size = min(int(batch_size), self.n_total)
        self.rng = np.random.default_rng(seed)

    def draw(self):
        if self.batch_size >= self.n_total:
            return np.arange(self.n_total, dtype=int)
        return self.rng.choice(self.n_total, size=self.batch_size, replace=False)


## 3. Initial-calibration trainer and crash-safe evaluation layer

The common crash-safe trainer below is configured in two distinct ways. **Stage I** preserves the historical v5 schedule exactly, including maturity-balanced sampling and its learning-rate milestones. **Stage II** keeps the 600-epoch architecture-transfer stage but switches to the final-v7 optimization conventions: an 80% split of the full filtered OTM surface, uniform 400-contract sampling, unbalanced contract-level raw-price MSE, raw-RMSE checkpoint selection, constant $10^{-4}$ NLV learning rate and $10^{-3}$ control-variate learning rate. Resumable checkpoints store the NumPy sampler and PyTorch RNG states.

Evaluation retains the complete crash-safe v7 execution design: one CPU thread, 2,000-path chunks, preallocated `float32` terminal arrays, sequential AM/PM settlement groups, explicit deletion/garbage collection, simple elementwise Pearson reduction, and persistent native-fault/RSS logs. These safeguards do not alter the model or estimator.


In [6]:

@dataclass(frozen=True)
class LegacyStageConfig:
    tag: str
    epochs: int
    milestones: tuple[int, int] | None
    balanced_loss: bool
    monitor_field: str
    seed: int = SEED
    sampler_mode: str = "maturity_balanced"
    contracts_per_step: int = 400
    sde_lr: float = LEGACY_LR
    hedge_lr: float = LEGACY_LR
    use_scheduler: bool = True


def legacy_stage_path(tag):
    return INITIAL_DIR / f"{tag}.pt"


def _legacy_checkpoint(path, *, model, hedge, opt_sde, opt_hedge, scheduler,
                       sampler, history, candidates, next_epoch, completed=False,
                       final_model_state=None, rerank=None):
    atomic_save({
        "completed": completed,
        "next_epoch": int(next_epoch),
        "model_state": cpu_state(model),
        "hedge_state": cpu_state(hedge),
        "opt_sde": opt_sde.state_dict(),
        "opt_hedge": opt_hedge.state_dict(),
        "scheduler": None if scheduler is None else scheduler.state_dict(),
        "sampler_rng": sampler.rng.bit_generator.state,
        "torch_rng": torch.get_rng_state(),
        "history": history,
        "candidates": candidates,
        "final_model_state": final_model_state,
        "rerank": rerank,
    }, path)


def train_legacy_stage(ctx, model, hedge, cfg):
    """Crash-safe alternating trainer; Stage I uses legacy v5 settings, Stage II uses final-v7 settings."""
    path = legacy_stage_path(cfg.tag)
    opt_sde = torch.optim.Adam(model.parameters(), lr=cfg.sde_lr)
    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=cfg.hedge_lr)
    scheduler = (
        torch.optim.lr_scheduler.MultiStepLR(opt_sde, milestones=list(cfg.milestones), gamma=0.2)
        if cfg.use_scheduler else None
    )
    if cfg.sampler_mode == "maturity_balanced":
        sampler = MaturityBalancedSampler(ctx, LEGACY_QUOTES_PER_MATURITY, SEED + 10)
    elif cfg.sampler_mode == "uniform":
        sampler = UniformContractSampler(ctx, cfg.contracts_per_step, SEED + 10)
    else:
        raise ValueError(f"Unknown sampler_mode={cfg.sampler_mode!r}")
    z_monitor = antithetic_normals(LEGACY_MONITOR_PATHS, len(ctx.grid_np)-1, SEED + 100)

    history, candidates, start_epoch = [], [], 0
    if path.exists():
        p = safe_torch_load(path)
        if p.get("completed", False):
            model.load_state_dict(p["final_model_state"], strict=True)
            model.eval(); set_requires_grad(model, False)
            return model, pd.DataFrame(p["history"]), pd.DataFrame(p["rerank"])
        model.load_state_dict(p["model_state"], strict=True)
        hedge.load_state_dict(p["hedge_state"], strict=True)
        opt_sde.load_state_dict(p["opt_sde"]); opt_hedge.load_state_dict(p["opt_hedge"])
        if scheduler is not None and p.get("scheduler") is not None:
            scheduler.load_state_dict(p["scheduler"])
        sampler.rng.bit_generator.state = p["sampler_rng"]
        torch.set_rng_state(p["torch_rng"])
        history, candidates, start_epoch = p["history"], p["candidates"], int(p["next_epoch"])
        print(f"[resume] {cfg.tag}: epoch {start_epoch+1}/{cfg.epochs}")

    sde_count = sum(e % 2 == 0 for e in range(start_epoch))
    bar = tqdm(range(start_epoch, cfg.epochs), desc=cfg.tag, unit="epoch", dynamic_ncols=True)

    for epoch in bar:
        phase = "sde" if epoch % 2 == 0 else "hedge"
        set_requires_grad(model, phase == "sde")
        set_requires_grad(hedge, phase == "hedge")
        losses = []

        for _ in range(LEGACY_STEPS_PER_EPOCH):
            ids = sampler.draw()
            z = torch.randn(LEGACY_PATHS, len(ctx.grid_np)-1, dtype=torch.float32, device=DEVICE)
            opt_sde.zero_grad(set_to_none=True); opt_hedge.zero_grad(set_to_none=True)
            _, cv = simulate_estimators(model, hedge, ctx, ids, z, phase)
            ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)

            if phase == "sde":
                residual = (cv.mean(0) - ctx.target[ids_t]) / ctx.spot
                loss = maturity_balanced_mean(residual.square(), ctx.stage_tensor[ids_t]) if cfg.balanced_loss else residual.square().mean()
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt_sde.step()
            else:
                var = torch.var(cv / ctx.spot, dim=0, unbiased=False)
                loss = maturity_balanced_mean(var, ctx.stage_tensor[ids_t]) if cfg.balanced_loss else var.mean()
                loss.backward(); nn.utils.clip_grad_norm_(hedge.parameters(), 3.0); opt_hedge.step()

            losses.append(float(loss.detach().cpu()))
            del z, cv, loss

        if scheduler is not None:
            scheduler.step()
        diag = None
        if phase == "sde":
            sde_count += 1
            if sde_count == 1 or sde_count % LEGACY_EVAL_EVERY_SDE == 0:
                set_requires_grad(model, False); set_requires_grad(hedge, False)
                diag = objective_diagnostics(model, ctx, z_monitor)
                score = float(diag[cfg.monitor_field])
                candidates.append({
                    "epoch": epoch + 1,
                    "monitor_score": score,
                    "raw_rmse": float(diag["raw_rmse"]),
                    "state": cpu_state(model),
                })
                candidates = sorted(candidates, key=lambda x: x["monitor_score"])[:LEGACY_TOP_K]

        history.append({
            "epoch": epoch + 1, "phase": phase, "train_loss": float(np.mean(losses)),
            "sde_lr": float(opt_sde.param_groups[0]["lr"]),
            "monitor_score": np.nan if diag is None else float(diag[cfg.monitor_field]),
            "raw_rmse": np.nan if diag is None else float(diag["raw_rmse"]),
        })
        if diag is not None:
            bar.set_postfix(raw=f"{diag['raw_rmse']:.3f}", lr=f"{opt_sde.param_groups[0]['lr']:.1e}")

        if (epoch + 1) % LEGACY_SAVE_EVERY == 0 or epoch == cfg.epochs - 1:
            _legacy_checkpoint(
                path, model=model, hedge=hedge, opt_sde=opt_sde, opt_hedge=opt_hedge,
                scheduler=scheduler, sampler=sampler, history=history, candidates=candidates,
                next_epoch=epoch + 1
            )
            pd.DataFrame(history).to_csv(INITIAL_DIR / f"{cfg.tag}_history.csv", index=False)
        gc.collect()

    # Independent 50k rerank, exactly as in v5/v6.
    z = antithetic_normals(LEGACY_RERANK_PATHS, len(ctx.grid_np)-1, SEED + 5000)
    temp = type(model)(ctx.maturity_boundaries).to(DEVICE) if isinstance(model, MaturityHeadNeuralLocalVol) else SharedNeuralLocalVol().to(DEVICE)
    rows = []
    for c in candidates:
        temp.load_state_dict(c["state"], strict=True)
        d = objective_diagnostics(temp, ctx, z)
        rows.append({
            "epoch": c["epoch"], "monitor_score": c["monitor_score"],
            "rerank_score": float(d[cfg.monitor_field]), "rerank_raw_rmse": float(d["raw_rmse"])
        })
    rerank = pd.DataFrame(rows).sort_values(["rerank_score", "monitor_score"]).reset_index(drop=True)
    best_epoch = int(rerank.iloc[0]["epoch"])
    best = next(c for c in candidates if c["epoch"] == best_epoch)
    model.load_state_dict(best["state"], strict=True)
    model.eval(); set_requires_grad(model, False)

    _legacy_checkpoint(
        path, model=model, hedge=hedge, opt_sde=opt_sde, opt_hedge=opt_hedge,
        scheduler=scheduler, sampler=sampler, history=history, candidates=[],
        next_epoch=cfg.epochs, completed=True, final_model_state=cpu_state(model),
        rerank=rerank.to_dict("records")
    )
    rerank.to_csv(INITIAL_DIR / f"{cfg.tag}_rerank.csv", index=False)
    del z, temp, hedge, opt_sde, opt_hedge
    gc.collect()
    print(f"{cfg.tag}: selected epoch {best_epoch}, 50k raw RMSE={rerank.iloc[0]['rerank_raw_rmse']:.4f}")
    return model, pd.DataFrame(history), rerank


In [7]:
def tkey(t):
    return round(float(t), 12)


def bs_forward_price(F, K, D, T, sigma, cp="C"):
    F, K, D, T, sigma = map(float, (F, K, D, T, sigma))
    if T <= 0 or sigma <= 0:
        return D*max(F-K, 0.0) if cp == "C" else D*max(K-F, 0.0)
    srt = sigma * math.sqrt(T)
    d1 = (math.log(F/K) + 0.5*srt*srt) / srt
    d2 = d1 - srt
    call = D * (F*float(ndtr(d1)) - K*float(ndtr(d2)))
    return call if cp == "C" else call - D*(F-K)


def implied_vol_from_price(price, F, K, D, T, cp):
    intrinsic = D*max(F-K, 0.0) if cp == "C" else D*max(K-F, 0.0)
    upper = D*F if cp == "C" else D*K
    if not (intrinsic-1e-8 <= price <= upper+1e-8) or T <= 0:
        return np.nan
    try:
        return float(brentq(
            lambda s: bs_forward_price(F, K, D, T, s, cp) - price,
            1e-5, 5.0, maxiter=100
        ))
    except ValueError:
        return np.nan


@torch.no_grad()
def simulate_same_day_x(model, maturities, total_paths, chunk_paths, seed, progress=True):
    """Low-memory antithetic MC with preallocated float32 terminal arrays."""
    if total_paths % 2:
        raise ValueError("total_paths must be even for antithetic evaluation.")

    grid = build_time_grid(maturities)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    total_pairs = total_paths // 2
    chunk_pairs = max(1, chunk_paths // 2)
    collected = {key: np.empty((total_pairs, 2), dtype=np.float32) for key in maturity_idx}

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    done = 0
    bar = tqdm(total=total_paths, desc="Same-day MC", unit="path", disable=not progress)

    while done < total_pairs:
        n_pairs = min(chunk_pairs, total_pairs-done)
        z_half = torch.randn(n_pairs, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], 0).to(DEVICE)
        x = torch.ones((2*n_pairs, 1), dtype=torch.float32, device=DEVICE)

        for step in range(len(grid)-1):
            t = float(grid[step])
            dt = float(grid[step+1] - grid[step])
            sigma = model(torch.full_like(x, t), x)
            dW = math.sqrt(dt) * z[:, step:step+1]
            a = x * sigma
            x = x + a*dW / (1.0 + torch.abs(a)*math.sqrt(dt))

            if step+1 in index_to_keys:
                arr = x.squeeze(1).detach().cpu().numpy()
                lo, hi = done, done+n_pairs
                for key in index_to_keys[step+1]:
                    collected[key][lo:hi, 0] = arr[:n_pairs]
                    collected[key][lo:hi, 1] = arr[n_pairs:]

        done += n_pairs
        bar.update(2*n_pairs)
        del z_half, z, x
        gc.collect()

    bar.close()
    return collected


def price_rows_same_day(rows, samples_by_T, base_col="mid"):
    out = rows.copy().reset_index(drop=True)
    price = np.empty(len(out))

    for T, idx in out.groupby("T").groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        samples = samples_by_T[tkey(T)].reshape(-1)
        payoff = empirical_payoff_mean(
            samples,
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["cp_flag"].to_numpy(),
        )
        price[pos] = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff

    out["neural_lv_price"] = price
    out["pricing_error"] = price - out[base_col].to_numpy(float)
    out["inside_spread"] = out["neural_lv_price"].between(out["bid"], out["ask"])
    return out


def add_model_iv(rows, desc="Model IV"):
    out = rows.copy()
    iv = [
        implied_vol_from_price(r.neural_lv_price, r.F, r.K, r.D, r.T, r.cp_flag)
        for r in tqdm(out.itertuples(index=False), total=len(out), desc=desc, unit="opt")
    ]
    out["iv_model"] = iv
    market_col = "iv_market" if "iv_market" in out.columns else "iv_market_otm"
    out["iv_error"] = out["iv_model"] - out[market_col].to_numpy(float)
    return out


def delta_rows_from_terminal(rows, samples_by_T, bump_bps=DELTA_BUMP_BPS):
    out = rows.copy().reset_index(drop=True)

    for bps in bump_bps:
        eps = float(bps) * 1e-4
        delta = np.empty(len(out))

        for T, idx in out.groupby("T").groups.items():
            pos = np.asarray(list(idx), dtype=int)
            g = out.loc[idx]
            samples = samples_by_T[tkey(T)].reshape(-1)

            F = g["F"].to_numpy(float)
            K = g["K"].to_numpy(float)
            D = g["D"].to_numpy(float)
            cp = g["cp_flag"].to_numpy()
            S = g["S"].to_numpy(float)

            Fu, Fd = F*(1+eps), F*(1-eps)
            pu = D*Fu*empirical_payoff_mean(samples, K/Fu, cp)
            pdn = D*Fd*empirical_payoff_mean(samples, K/Fd, cp)
            delta[pos] = (pu-pdn) / (2*eps*S)

        out[f"delta_{bps}bp"] = delta

    headline = f"delta_{DELTA_HEADLINE_BPS}bp"
    out["neural_lv_delta"] = out[headline]
    others = [f"delta_{b}bp" for b in bump_bps if b != DELTA_HEADLINE_BPS]
    out["delta_bump_instability"] = np.max(
        np.column_stack([np.abs(out[c]-out[headline]) for c in others]), axis=1
    )
    return out


def price_summary(df, error_col="pricing_error", market_col="mid", inside_col="inside_spread"):
    e = df[error_col].to_numpy(float)
    market = np.maximum(df[market_col].to_numpy(float), 1e-12)

    out = {
        "n": len(df),
        "MSE": float(np.mean(e**2)),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "MAE": float(np.mean(np.abs(e))),
        "Rel_MAE_pct": float(np.mean(np.abs(e)/market) * 100.0),
    }
    if inside_col in df:
        out["inside_bid_ask"] = float(df[inside_col].mean())
    if "iv_error" in df:
        ie = df["iv_error"].dropna().to_numpy(float)
        out["IV_RMSE"] = float(np.sqrt(np.mean(ie**2))) if len(ie) else np.nan
    return out


def evaluate_transition_state(model, date, paths=FINAL_MC_PATHS, seed=SEED+7500):
    """Current-day full-chain prices/deltas for the model actually used to forecast tomorrow."""
    date = pd.Timestamp(date)
    full = quotes_all.loc[quotes_all["date"] == date].copy().reset_index(drop=True)
    samples = simulate_same_day_x(
        model, full["T"].unique(), paths, FINAL_PATH_CHUNK, seed
    )
    out = delta_rows_from_terminal(price_rows_same_day(full, samples), samples)
    del samples
    gc.collect()
    return out

In [8]:
def build_evaluation_pairs(quotes):
    dates = sorted(quotes["date"].unique())
    next_map = {dates[i]: dates[i+1] for i in range(len(dates)-1)}
    left = quotes.copy()
    left["next_date"] = left["date"].map(next_map)

    cols = [
        "date", "optionid", "cp_flag", "K", "exdate", "am_settlement",
        "mid", "bid", "ask", "S", "F", "D", "r", "q", "carry", "T", "k",
        "delta_market",
    ]
    right = quotes[cols].rename(columns={
        "date": "next_date", "mid": "mid_next", "bid": "bid_next",
        "ask": "ask_next", "S": "S_next", "F": "F_next", "D": "D_next",
        "r": "r_next", "q": "q_next", "carry": "carry_next",
        "T": "T_next", "k": "k_next", "delta_market": "delta_market_next",
    })
    return left.merge(
        right,
        on=["next_date", "optionid", "cp_flag", "K", "exdate", "am_settlement"],
        how="inner",
        validate="many_to_one",
    )


@torch.no_grad()
def simulate_frozen_to_date(model, curve0, curve1, elapsed, maturities_remaining,
                            total_paths, chunk_paths, seed, progress=True):
    """Low-memory future-date MC with direct writes into preallocated float32 arrays."""
    if total_paths % 2:
        raise ValueError("total_paths must be even.")

    grid = build_time_grid(maturities_remaining)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities_remaining, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    total_pairs = total_paths // 2
    chunk_pairs = max(1, chunk_paths // 2)
    collected = {key: np.empty((total_pairs, 2), dtype=np.float32) for key in maturity_idx}

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    done = 0
    bar = tqdm(
        total=total_paths,
        desc=f"Future-date MC AM={curve1.am_settlement}",
        unit="path",
        disable=not progress,
    )

    while done < total_pairs:
        n_pairs = min(chunk_pairs, total_pairs-done)
        z_half = torch.randn(n_pairs, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], 0).to(DEVICE)
        y = torch.ones((2*n_pairs, 1), dtype=torch.float32, device=DEVICE)

        for step in range(len(grid)-1):
            v = float(grid[step])
            dt = float(grid[step+1] - grid[step])
            abs_t = elapsed + v

            Fnew = float(curve1.forward(v))
            Fold = float(curve0.forward(abs_t))
            x_old = y * (Fnew/Fold)

            sigma = model(torch.full_like(y, abs_t), x_old)
            dW = math.sqrt(dt) * z[:, step:step+1]
            a = y * sigma
            y = y + a*dW / (1.0 + torch.abs(a)*math.sqrt(dt))

            if step+1 in index_to_keys:
                arr = y.squeeze(1).detach().cpu().numpy()
                lo, hi = done, done+n_pairs
                for key in index_to_keys[step+1]:
                    collected[key][lo:hi, 0] = arr[:n_pairs]
                    collected[key][lo:hi, 1] = arr[n_pairs:]

        done += n_pairs
        bar.update(2*n_pairs)
        del z_half, z, y
        gc.collect()

    bar.close()
    return collected


def price_future_rows(rows, samples_by_am, price_col="model_price_future"):
    out = rows.copy().reset_index(drop=True)
    price = np.empty(len(out))

    for (am, T), idx in out.groupby(["am_settlement", "T"]).groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        samples = samples_by_am[int(am)][tkey(T)].reshape(-1)
        payoff = empirical_payoff_mean(
            samples,
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["cp_flag"].to_numpy(),
        )
        price[pos] = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff

    out[price_col] = price
    out["pricing_error"] = price - out["mid"].to_numpy(float)
    out["inside_spread"] = out[price_col].between(out["bid"], out["ask"])
    return out


def pearson_reduction(x, y):
    """Pearson correlation via elementwise reductions; avoids the corrcoef/dot route.

    This is a diagnostic workaround, not evidence that BLAS caused the crash.
    Inputs are paired after dropping NaNs, as in the existing summary.
    """
    x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)
    if x.size < 2 or not (np.isfinite(x).all() and np.isfinite(y).all()):
        return np.nan
    x, y = x - x.mean(), y - y.mean()
    sx, sy = float(np.max(np.abs(x))), float(np.max(np.abs(y)))
    if sx == 0 or sy == 0:
        return np.nan
    x, y = x / sx, y / sy
    denominator = math.sqrt(float(np.sum(x*x))) * math.sqrt(float(np.sum(y*y)))
    return float(np.clip(np.sum(x*y) / denominator, -1.0, 1.0))


def _transition_subset_stats(df):
    if len(df) == 0:
        return {
            "n": 0,
            "pricing_RMSE": np.nan, "pricing_MAE": np.nan, "pricing_Rel_MAE_pct": np.nan,
            "pricing_inside_bid_ask": np.nan,
            "NLV_hedge_RMSE": np.nan, "NLV_hedge_MAE": np.nan, "NLV_hedge_bias": np.nan,
            "market_delta_hedge_RMSE": np.nan, "market_delta_hedge_MAE": np.nan,
            "market_delta_hedge_bias": np.nan,
            "NLV_minus_market_delta_RMSE": np.nan, "delta_correlation": np.nan,
        }

    pe = df["next_day_pricing_error"].to_numpy(float)
    nlv_he = df["hedge_error"].dropna().to_numpy(float)
    mkt_he = df["market_delta_hedge_error"].dropna().to_numpy(float)

    good_delta = df[["neural_lv_delta", "delta_market"]].dropna()
    dd = (good_delta["neural_lv_delta"] - good_delta["delta_market"]).to_numpy(float)

    return {
        "n": len(df),
        "pricing_RMSE": float(np.sqrt(np.mean(pe**2))),
        "pricing_MAE": float(np.mean(np.abs(pe))),
        "pricing_Rel_MAE_pct": float(
            np.mean(np.abs(pe)/np.maximum(df["mid_next"].to_numpy(float), 1e-12))*100
        ),
        "pricing_inside_bid_ask": float(df["next_day_inside_bid_ask"].mean()),
        "NLV_hedge_RMSE": float(np.sqrt(np.mean(nlv_he**2))) if len(nlv_he) else np.nan,
        "NLV_hedge_MAE": float(np.mean(np.abs(nlv_he))) if len(nlv_he) else np.nan,
        "NLV_hedge_bias": float(np.mean(nlv_he)) if len(nlv_he) else np.nan,
        "market_delta_hedge_RMSE": float(np.sqrt(np.mean(mkt_he**2))) if len(mkt_he) else np.nan,
        "market_delta_hedge_MAE": float(np.mean(np.abs(mkt_he))) if len(mkt_he) else np.nan,
        "market_delta_hedge_bias": float(np.mean(mkt_he)) if len(mkt_he) else np.nan,
        "NLV_minus_market_delta_RMSE": float(np.sqrt(np.mean(dd**2))) if len(dd) else np.nan,
        "delta_correlation": pearson_reduction(good_delta["neural_lv_delta"], good_delta["delta_market"])
            if len(good_delta) > 1 else np.nan,
    }


def evaluate_transition(model, date, next_date, same_day_full, paths=FINAL_MC_PATHS, seed=SEED+8000):
    """Low-memory next-day evaluation: one settlement group at a time."""
    stage_log("transition: begin")
    date, next_date = pd.Timestamp(date), pd.Timestamp(next_date)
    pairs = evaluation_pairs.loc[
        (evaluation_pairs["date"] == date) & (evaluation_pairs["next_date"] == next_date)
    ].copy().reset_index(drop=True)

    elapsed = (next_date-date).days / 365.0
    priced_pieces = []

    for am in sorted(pairs["am_settlement"].unique()):
        g = pairs.loc[pairs["am_settlement"] == am].copy().reset_index(drop=True)
        stage_log(f"transition AM={am}: simulation begin, rows={len(g)}")
        samples = simulate_frozen_to_date(
            model,
            curve_book.get(date, int(am)),
            curve_book.get(next_date, int(am)),
            elapsed,
            g["T_next"].unique(),
            paths,
            FINAL_PATH_CHUNK,
            seed + int(am),
        )

        stage_log(f"transition AM={am}: simulation returned; terminal storage={sum(v.nbytes for v in samples.values())/2**20:.1f} MiB")
        price = np.empty(len(g), dtype=np.float64)
        for T, idx in g.groupby("T_next").groups.items():
            pos = np.asarray(list(idx), dtype=int)
            gg = g.loc[idx]
            terminal = samples[tkey(T)].reshape(-1)
            stage_log(f"transition AM={am}: payoff begin T={T}, contracts={len(gg)}")
            payoff = empirical_payoff_mean(
                terminal,
                gg["K"].to_numpy(float) / gg["F_next"].to_numpy(float),
                gg["cp_flag"].to_numpy(),
            )
            price[pos] = gg["D_next"].to_numpy(float) * gg["F_next"].to_numpy(float) * payoff

        stage_log(f"transition AM={am}: pricing complete")
        del terminal, gg, payoff
        g["model_price_next"] = price
        priced_pieces.append(g)
        del samples, price, g
        gc.collect()

    stage_log("transition: concatenate begin")
    out = pd.concat(priced_pieces, ignore_index=True)
    del priced_pieces, pairs
    gc.collect()

    out["next_day_pricing_error"] = out["model_price_next"] - out["mid_next"].to_numpy(float)
    out["next_day_inside_bid_ask"] = out["model_price_next"].between(out["bid_next"], out["ask_next"])

    stage_log("transition: delta merge begin")
    delta_map = same_day_full[
        ["optionid", "neural_lv_delta", "delta_bump_instability"]
    ].drop_duplicates("optionid")
    out = out.merge(delta_map, on="optionid", how="left", validate="many_to_one")
    out["dP_market"] = out["mid_next"] - out["mid"]
    out["dS"] = out["S_next"] - out["S"]

    # Keep the old delta residual only as a diagnostic. Headline hedge errors use the
    # same one-period self-financing accounting that should be applied to every baseline:
    #   dC - delta*dS - r*(C-delta*S)*dt - q*delta*S*dt.
    # The actual calendar gap is used for dt.
    out["simple_hedge_error"] = out["dP_market"] - out["neural_lv_delta"]*out["dS"]
    out["simple_market_delta_hedge_error"] = out["dP_market"] - out["delta_market"]*out["dS"]

    cash_nlv = out["mid"] - out["neural_lv_delta"]*out["S"]
    cash_mkt = out["mid"] - out["delta_market"]*out["S"]
    out["hedge_error"] = (
        out["simple_hedge_error"]
        - out["r"]*cash_nlv*elapsed
        - out["q"]*out["neural_lv_delta"]*out["S"]*elapsed
    )
    out["market_delta_hedge_error"] = (
        out["simple_market_delta_hedge_error"]
        - out["r"]*cash_mkt*elapsed
        - out["q"]*out["delta_market"]*out["S"]*elapsed
    )

    stage_log("transition: hedge calculations complete; OTM flags begin")
    current_otm_ids = otm_option_ids_by_date.get(date, set())
    next_otm_ids = otm_option_ids_by_date.get(next_date, set())
    ids = out["optionid"].astype(int)
    out["current_is_otm"] = ids.isin(current_otm_ids)
    out["next_is_otm"] = ids.isin(next_otm_ids)
    out["otm_on_both_dates"] = out["current_is_otm"] & out["next_is_otm"]

    stage_log("transition: full summary begin")
    full_stats = _transition_subset_stats(out)
    stage_log("transition: OTM summary begin")
    otm_stats = _transition_subset_stats(out.loc[out["otm_on_both_dates"]].copy())

    summary = {"date": date, "next_date": next_date}
    summary.update({f"full_{k}": v for k, v in full_stats.items()})
    summary.update({f"otm_{k}": v for k, v in otm_stats.items()})
    gc.collect()
    stage_log("transition: complete")
    return summary, out


evaluation_pairs = build_evaluation_pairs(quotes_all)

def pooled_summary(frame):
    n = frame["otm_n"].to_numpy(float)
    total = n.sum()
    return {
        "n": int(total),
        "pricing_RMSE": float(np.sqrt(np.sum(n * frame["otm_pricing_RMSE"].to_numpy(float)**2) / total)),
        "pricing_MAE": float(np.sum(n * frame["otm_pricing_MAE"].to_numpy(float)) / total),
        "NLV_hedge_RMSE": float(np.sqrt(np.sum(n * frame["otm_NLV_hedge_RMSE"].to_numpy(float)**2) / total)),
        "market_delta_hedge_RMSE": float(np.sqrt(np.sum(n * frame["otm_market_delta_hedge_RMSE"].to_numpy(float)**2) / total)),
    }

def evaluate_cached(model, date, next_date, *, index, paths, folder):
    """Crash-safe common-random-number evaluation with immediate per-transition persistence."""
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    stamp = f"{pd.Timestamp(date):%Y%m%d}"
    summary_path, contract_path = folder / f"{stamp}_summary.csv", folder / f"{stamp}_contracts.csv"
    if summary_path.exists() and contract_path.exists():
        return pd.read_csv(summary_path).iloc[0].to_dict()

    state = evaluate_transition_state(
        model, date, paths=paths, seed=SEED + 1_200_000 + int(index) * 100
    )
    summary, contracts = evaluate_transition(
        model, date, next_date, state, paths=paths,
        seed=SEED + 1_300_000 + int(index) * 100
    )
    pd.DataFrame([summary]).to_csv(summary_path, index=False)
    contracts.to_csv(contract_path, index=False)
    del state, contracts
    gc.collect()
    stage_log(f"Saved transition {pd.Timestamp(date).date()} -> {pd.Timestamp(next_date).date()} ({paths:,} paths)")
    return summary


In [9]:
def bs_delta_unit_test():
    class ConstantVol(nn.Module):
        def __init__(self, sigma):
            super().__init__()
            self.sigma = float(sigma)
        def forward(self, t, x):
            return torch.full_like(x, self.sigma)

    S, r, q, T, K, sig = 5000.0, 0.04, 0.015, 0.5, 5100.0, 0.20
    F = S*math.exp((r-q)*T)
    D = math.exp(-r*T)

    samples = simulate_same_day_x(
        ConstantVol(sig).to(DEVICE), [T], 30_000, 10_000, SEED+6600, progress=False
    )
    row = pd.DataFrame([{"S": S, "F": F, "D": D, "K": K, "T": T, "cp_flag": "C"}])
    mc = float(delta_rows_from_terminal(row, samples)["neural_lv_delta"].iloc[0])

    d1 = (math.log(S/K)+(r-q+0.5*sig*sig)*T)/(sig*math.sqrt(T))
    exact = math.exp(-q*T)*float(ndtr(d1))
    err = abs(mc-exact)

    print(f"BS delta unit test: MC={mc:.5f}, analytic={exact:.5f}, abs error={err:.5f}")
    if err > 0.03:
        warnings.warn("Delta unit test is noisier than expected.")
    return err


def clock_unit_test():
    class ToyCurve:
        am_settlement = 0
        def __init__(self, spot, carry):
            self.spot, self.carry = float(spot), float(carry)
        def forward(self, t):
            return self.spot*np.exp(self.carry*np.asarray(t, float))

    class Recorder(nn.Module):
        def __init__(self):
            super().__init__()
            self.times = []
        def forward(self, t, x):
            self.times.append(float(t.reshape(-1)[0]))
            return torch.full_like(x, 0.2)

    elapsed = 0.10
    rec = Recorder().to(DEVICE)
    simulate_frozen_to_date(
        rec, ToyCurve(100, 0.02), ToyCurve(101, 0.02),
        elapsed, [0.08], 4, 4, SEED+6700, progress=False
    )
    err = abs(rec.times[0] - elapsed)
    print(f"Clock unit test error={err:.2e}")
    if err > 1e-6:
        raise RuntimeError("Frozen model clock restarted incorrectly.")
    return err


delta_test = bs_delta_unit_test()
clock_test = clock_unit_test()
shift_test = trunk_time_shift_unit_test()
print({
    "Black-Scholes delta absolute error": delta_test,
    "future-date clock absolute error": clock_test,
    "calendar-time transfer max error": shift_test,
})



stage_log("Initial calibration I: v5 shared Neural-LV")
ctx_v5 = build_calibration_context(INITIAL_DATE, initial_capped)
set_seed(SEED)
shared = SharedNeuralLocalVol().to(DEVICE)
cv_v5 = IntervalHedgeBank(ctx_v5.maturity_boundaries, len(ctx_v5.contracts)).to(DEVICE)
shared, v5_history, v5_rerank = train_legacy_stage(
    ctx_v5, shared, cv_v5,
    LegacyStageConfig("initial_shared_v5", V5_EPOCHS, (500, 800), False, "raw_rmse")
)

stage_log("Initial calibration II: maturity-head transfer under final-v7 objective")
ctx_v6 = build_calibration_context(INITIAL_DATE, initial_train80)
# Explicit seed removes the incidental optional-cell RNG dependence of the exploratory v6 notebook.
set_seed(SEED)
headed = MaturityHeadNeuralLocalVol(ctx_v6.maturity_boundaries).to(DEVICE)
initialise_maturity_heads_from_shared(shared, headed)
cv_v6 = IntervalHedgeBank(ctx_v6.maturity_boundaries, len(ctx_v6.contracts)).to(DEVICE)
headed, v6_history, v6_rerank = train_legacy_stage(
    ctx_v6, headed, cv_v6,
    LegacyStageConfig(
        "initial_maturity_heads_v7spec", V6_HEAD_EPOCHS, None, False, "raw_rmse",
        sampler_mode="uniform", contracts_per_step=400,
        sde_lr=1e-4, hedge_lr=1e-3, use_scheduler=False,
    )
)

# Same-day held-out diagnostic for the Stage-II maturity-head model (20% of full OTM).
full_mar1 = quotes_all.loc[quotes_all["date"] == INITIAL_DATE].copy().reset_index(drop=True)
samples = simulate_same_day_x(headed, full_mar1["T"].unique(), PRIMARY_EVAL_PATHS, FINAL_PATH_CHUNK, SEED + 7000)
priced = add_model_iv(price_rows_same_day(full_mar1, samples), "v6-head IV")
train_ids, hold_ids = set(initial_train80["optionid"].astype(int)), set(initial_holdout20["optionid"].astype(int))
v6_initial_summary = pd.DataFrame({
    "train80": price_summary(priced.loc[priced["optionid"].astype(int).isin(train_ids)]),
    "holdout20": price_summary(priced.loc[priced["optionid"].astype(int).isin(hold_ids)]),
}).T
display(v6_initial_summary)
v6_initial_summary.to_csv(RESULT_DIR / "initial_v6_holdout_summary.csv")
del samples, priced, full_mar1, cv_v5, cv_v6, ctx_v5
gc.collect()


BS delta unit test: MC=0.50156, analytic=0.50382, abs error=0.00226
Clock unit test error=1.49e-09
Warm-start 2024-03-01 -> 2024-03-04: 3 heads | elapsed=0.00822y | median head mismatch=0.00178y
Trunk time-shift unit test max error=3.58e-07
{'Black-Scholes delta absolute error': 0.00225702690985774, 'future-date clock absolute error': 1.4901161138336505e-09, 'calendar-time transfer max error': 3.5762786865234375e-07}
2026-09-24T16:37:08 | Initial calibration I: v5 shared Neural-LV | RSS=581 MiB


initial_shared_v5:   0%|          | 0/1000 [00:00<?, ?epoch/s]

initial_shared_v5: selected epoch 879, 50k raw RMSE=2.1346
2026-09-25T02:54:26 | Initial calibration II: maturity-head transfer under final-v7 objective | RSS=555 MiB


initial_maturity_heads_v7spec:   0%|          | 0/600 [00:00<?, ?epoch/s]

initial_maturity_heads_v7spec: selected epoch 539, 50k raw RMSE=1.9561


Same-day MC:   0%|          | 0/100000 [00:00<?, ?path/s]

v6-head IV:   0%|          | 0/9046 [00:00<?, ?opt/s]

,n,MSE,RMSE,MAE,Rel_MAE_pct,inside_bid_ask,IV_RMSE
train80,3443.0,3.137692,1.771353,1.284689,4.606396,0.127505,0.003975
holdout20,861.0,3.041508,1.743992,1.283687,4.672074,0.119628,0.003976


606


## 4. Complete the 1-March initial calibration

The v6 maturity-head stage above is followed by the v7 crash-safe full-sample refinement. The inherited model at update 0 can remain the final model; otherwise refinement runs for at most 30 NLV updates, with minimum 10 and patience 12, and update 0 plus the best remaining fixed-monitor checkpoints (ten candidates in total) are reranked on an independent 50k-path sample. Update 0 is never pruned before this rerank.

Once the 1-March model is fixed, all large intermediate objects are released before rolling evaluation.


In [10]:

@dataclass(frozen=True)
class RefinementConfig:
    max_updates: int = 30
    min_updates: int = 10
    patience: int = 12
    paths: int = 4_096
    steps_per_phase: int = 10
    contracts_per_step: int = 400
    monitor_paths: int = 16_384
    top_k: int = 10
    rerank_paths: int = 50_000
    sde_lr: float = 1e-4
    hedge_lr: float = 1e-3
    save_every: int = 5
    seed_offset: int = 50_000


def train_initial_full_sample_refinement(ctx, previous_model, previous_date, cfg=RefinementConfig()):
    """v7 crash-safe full-sample refinement; update 0 is eligible for final selection."""
    path = INITIAL_DIR / "initial_full_otm_refinement_20240301.pt"
    if path.exists():
        p = safe_torch_load(path)
        if p.get("completed", False):
            model = MaturityHeadNeuralLocalVol(p["boundaries"]).to(DEVICE)
            model.load_state_dict(p["final_model_state"], strict=True)
            model.eval(); set_requires_grad(model, False)
            return model, pd.DataFrame(p["history"]), pd.DataFrame(p["rerank"])
        del p

    set_seed(SEED + cfg.seed_offset)
    model = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE)
    transfer_rolling_model(previous_model, model, previous_date, ctx.date)
    hedge = IntervalHedgeBank(ctx.maturity_boundaries, len(ctx.contracts)).to(DEVICE)
    opt_sde = torch.optim.Adam(model.parameters(), lr=cfg.sde_lr)
    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=cfg.hedge_lr)
    sampler = UniformContractSampler(ctx, cfg.contracts_per_step, SEED + 10 + cfg.seed_offset)
    z_monitor = antithetic_normals(cfg.monitor_paths, len(ctx.grid_np)-1, SEED + 100 + cfg.seed_offset)

    history, candidates, next_update = [], [], 0
    best, bad = np.inf, 0
    if path.exists():
        p = safe_torch_load(path)
        model.load_state_dict(p["model_state"], strict=True); hedge.load_state_dict(p["hedge_state"], strict=True)
        opt_sde.load_state_dict(p["opt_sde"]); opt_hedge.load_state_dict(p["opt_hedge"])
        sampler.rng.bit_generator.state = p["sampler_rng"]; torch.set_rng_state(p["torch_rng"])
        history, candidates = p["history"], p["candidates"]
        next_update, best, bad = int(p["next_update"]), float(p["best"]), int(p["bad"])
        print(f"[resume] full-sample refinement: update {next_update+1}/{cfg.max_updates}")
    else:
        d0 = objective_diagnostics(model, ctx, z_monitor)
        best = float(d0["raw_rmse"])
        candidates = [{"update": 0, "monitor_rmse": best, "state": cpu_state(model)}]
        history = [{"update": 0, "sde_loss": np.nan, "hedge_loss": np.nan, "monitor_rmse": best, "bad": 0}]
        print(f"Initial full-sample raw RMSE before refinement: {best:.4f}")

    def save(completed=False, final=None, rerank=None):
        payload = {
            "completed": completed, "boundaries": ctx.maturity_boundaries,
            "history": history, "rerank": rerank,
            "final_model_state": final,
        }
        if not completed:
            payload.update({
                "next_update": next_update, "best": best, "bad": bad,
                "model_state": cpu_state(model), "hedge_state": cpu_state(hedge),
                "opt_sde": opt_sde.state_dict(), "opt_hedge": opt_hedge.state_dict(),
                "sampler_rng": sampler.rng.bit_generator.state, "torch_rng": torch.get_rng_state(),
                "candidates": candidates,
            })
        atomic_save(payload, path)

    for update in tqdm(range(next_update + 1, cfg.max_updates + 1), desc="initial full-sample refinement", unit="SDE"):
        model.train(); hedge.eval(); set_requires_grad(model, True); set_requires_grad(hedge, False)
        sde_losses = []
        for _ in range(cfg.steps_per_phase):
            ids = sampler.draw(); ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)
            z = torch.randn(cfg.paths, len(ctx.grid_np)-1, dtype=torch.float32, device=DEVICE)
            opt_sde.zero_grad(set_to_none=True)
            _, cv = simulate_estimators(model, hedge, ctx, ids, z, "sde")
            loss = ((cv.mean(0) - ctx.target[ids_t]) / ctx.spot).square().mean()
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt_sde.step()
            sde_losses.append(float(loss.detach().cpu())); del z, cv, loss

        model.eval(); set_requires_grad(model, False); set_requires_grad(hedge, False)
        score = float(objective_diagnostics(model, ctx, z_monitor)["raw_rmse"])
        candidates.append({"update": update, "monitor_rmse": score, "state": cpu_state(model)})
        # Update 0 is the inherited Stage-II anchor and must always reach the independent rerank.
        anchor = next((c for c in candidates if int(c["update"]) == 0), None)
        if anchor is None:
            raise RuntimeError("Stage-III update-0 anchor was lost from the candidate set.")
        non_anchor = sorted(
            (c for c in candidates if int(c["update"]) != 0),
            key=lambda x: x["monitor_rmse"],
        )[:max(cfg.top_k - 1, 0)]
        candidates = [anchor] + non_anchor
        if score < best: best, bad = score, 0
        else: bad += 1
        stop = update >= cfg.min_updates and bad >= cfg.patience

        hedge_loss = np.nan
        if not stop:
            model.eval(); hedge.train(); set_requires_grad(model, False); set_requires_grad(hedge, True)
            vals = []
            for _ in range(cfg.steps_per_phase):
                ids = sampler.draw()
                z = torch.randn(cfg.paths, len(ctx.grid_np)-1, dtype=torch.float32, device=DEVICE)
                opt_hedge.zero_grad(set_to_none=True)
                _, cv = simulate_estimators(model, hedge, ctx, ids, z, "hedge")
                loss = torch.var(cv / ctx.spot, dim=0, unbiased=False).mean()
                loss.backward(); nn.utils.clip_grad_norm_(hedge.parameters(), 3.0); opt_hedge.step()
                vals.append(float(loss.detach().cpu())); del z, cv, loss
            hedge_loss = float(np.mean(vals))

        history.append({"update": update, "sde_loss": np.mean(sde_losses), "hedge_loss": hedge_loss,
                        "monitor_rmse": score, "bad": bad})
        next_update = update
        if update % cfg.save_every == 0 or stop or update == cfg.max_updates:
            save(); pd.DataFrame(history).to_csv(INITIAL_DIR / "initial_full_otm_refinement_history.csv", index=False)
        if stop:
            print(f"Early stop at update {update}: no new monitor best for {cfg.patience} updates")
            break
        gc.collect()

    # Independent 50k rerank: update 0 plus the best remaining fixed-monitor checkpoints.
    if not any(int(c["update"]) == 0 for c in candidates):
        raise RuntimeError("Update 0 must be present in the Stage-III final rerank.")
    z = antithetic_normals(cfg.rerank_paths, len(ctx.grid_np)-1, SEED + 5000 + cfg.seed_offset)
    temp = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE).eval()
    rows = []
    for c in candidates:
        temp.load_state_dict(c["state"], strict=True)
        d = objective_diagnostics(temp, ctx, z)
        rows.append({"update": c["update"], "monitor_rmse": c["monitor_rmse"], "rerank_rmse": d["raw_rmse"]})
    rerank = pd.DataFrame(rows).sort_values(["rerank_rmse", "monitor_rmse"]).reset_index(drop=True)
    chosen = next(c for c in candidates if c["update"] == int(rerank.iloc[0]["update"]))
    model.load_state_dict(chosen["state"], strict=True); model.eval(); set_requires_grad(model, False)
    save(completed=True, final=cpu_state(model), rerank=rerank.to_dict("records"))
    rerank.to_csv(INITIAL_DIR / "initial_full_otm_refinement_rerank.csv", index=False)
    del z, temp, hedge, opt_sde, opt_hedge, candidates
    gc.collect()
    print(f"Full-sample refinement selected update {int(rerank.iloc[0]['update'])}, 50k raw RMSE={rerank.iloc[0]['rerank_rmse']:.4f}")
    return model, pd.DataFrame(history), rerank



stage_log("Initial calibration III: v7 crash-safe full-sample refinement")
ctx_full = build_calibration_context(INITIAL_DATE, initial_all_otm)
initial_model, refinement_history, refinement_rerank = train_initial_full_sample_refinement(
    ctx_full, headed, INITIAL_DATE
)

# Independent 100k same-day diagnostic of the final initial-date model.
initial_state = evaluate_transition_state(initial_model, INITIAL_DATE, paths=PRIMARY_EVAL_PATHS, seed=SEED + 7500)
initial_otm_state = initial_state.loc[
    initial_state["optionid"].astype(int).isin(otm_option_ids_by_date[INITIAL_DATE])
].copy()
print("Final 1-Mar full-OTM same-day fit:")
print(pd.Series(price_summary(initial_otm_state)).to_string())
initial_otm_state.to_csv(RESULT_DIR / "initial_20240301_same_day_100k.csv", index=False)

# Release everything not needed for the rolling sequence.
del ctx_v6, ctx_full, headed, shared, initial_otm_state, initial_state
gc.collect()
stage_log("Initial calibration complete; nonessential objects released")


2026-09-25T06:38:21 | Initial calibration III: v7 crash-safe full-sample refinement | RSS=390 MiB
Warm-start 2024-03-01 -> 2024-03-01: 36 heads | elapsed=0.00000y | median head mismatch=0.00000y
Initial full-sample raw RMSE before refinement: 2.4381


initial full-sample refinement:   0%|          | 0/30 [00:00<?, ?SDE/s]

Early stop at update 14: no new monitor best for 12 updates
Full-sample refinement selected update 0, 50k raw RMSE=1.7186


Same-day MC:   0%|          | 0/100000 [00:00<?, ?path/s]

Final 1-Mar full-OTM same-day fit:
n                 4304.000000
MSE                  3.311272
RMSE                 1.819690
MAE                  1.310021
Rel_MAE_pct          4.496581
inside_bid_ask       0.107342
2026-09-25T06:48:57 | Initial calibration complete; nonessential objects released | RSS=596 MiB



## 5. Fixed 15-update rolling recalibration

From 4 March onward there is no same-day model selection. Each date receives exactly 15 NLV updates, each followed by the auxiliary control-variate phase.

Each rolling control-variate network now receives an explicit **pre-construction initialization seed**, removing dependence on the ambient PyTorch RNG state. After construction, the original date-specific training seed is reset so the training-shock stream remains separately reproducible. Mid-date checkpoints persist optimiser, sampler and RNG states so a resumed date follows the same path as an uninterrupted run.


In [11]:

ROLLING_UPDATES = 15
ROLLING_PATHS = 4_096
ROLLING_STEPS_PER_PHASE = 10
ROLLING_CONTRACTS_PER_STEP = 400
ROLLING_SDE_LR, ROLLING_CV_LR = 1e-4, 1e-3
ROLLING_SAVE_EVERY = 2


def rolling_checkpoint(date):
    return ROLLING_DIR / f"rolling_{pd.Timestamp(date):%Y%m%d}.pt"


def rolling_progress(date):
    return ROLLING_DIR / f"rolling_{pd.Timestamp(date):%Y%m%d}_inprogress.pt"


def load_maturity_model(path):
    p = safe_torch_load(path, map_location="cpu")
    model = MaturityHeadNeuralLocalVol(p["boundaries"]).to(DEVICE)
    model.load_state_dict(p["final_model_state"], strict=True)
    model.eval(); set_requires_grad(model, False)
    return model


def train_fixed_rolling_day(date, previous_model, previous_date):
    """Frozen CPU rule: exactly 15 SDE updates; no early stopping or checkpoint selection."""
    date = pd.Timestamp(date)
    selected = all_otm_universe(date)
    ctx = build_calibration_context(date, selected)
    final_path, progress_path = rolling_checkpoint(date), rolling_progress(date)

    if final_path.exists():
        return load_maturity_model(final_path), ctx

    model = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE)
    transfer_rolling_model(previous_model, model, previous_date, date)

    # Seed the auxiliary network *before* construction so its initialization is reproducible
    # and independent of ambient RNG state. Use a separate seed from the training-shock stream.
    hedge_seed = SEED + 100_000 + int(date.day) * 17
    training_seed = SEED + int(date.day) * 17
    torch.manual_seed(hedge_seed)
    hedge = IntervalHedgeBank(ctx.maturity_boundaries, len(ctx.contracts)).to(DEVICE)

    opt_sde = torch.optim.Adam(model.parameters(), lr=ROLLING_SDE_LR)
    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=ROLLING_CV_LR)
    sampler = UniformContractSampler(ctx, ROLLING_CONTRACTS_PER_STEP, SEED + int(date.day) * 101)

    history, start_update = [], 0
    if progress_path.exists():
        p = safe_torch_load(progress_path)
        model.load_state_dict(p["model_state"], strict=True); hedge.load_state_dict(p["hedge_state"], strict=True)
        opt_sde.load_state_dict(p["opt_sde"]); opt_hedge.load_state_dict(p["opt_hedge"])
        sampler.rng.bit_generator.state = p["sampler_rng"]; torch.set_rng_state(p["torch_rng"])
        history, start_update = p["history"], int(p["last_update"])
        print(f"[resume] {date.date()}: update {start_update}/{ROLLING_UPDATES}")
    else:
        # Reset after deterministic CV initialization so the training-shock stream has its own seed.
        torch.manual_seed(training_seed)

    for update in tqdm(range(start_update + 1, ROLLING_UPDATES + 1), desc=f"rolling {date:%Y-%m-%d}", unit="SDE"):
        model.train(); hedge.eval(); set_requires_grad(model, True); set_requires_grad(hedge, False)
        sde_losses = []
        for _ in range(ROLLING_STEPS_PER_PHASE):
            ids = sampler.draw(); ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)
            z = torch.randn(ROLLING_PATHS, len(ctx.grid_np)-1, dtype=torch.float32, device=DEVICE)
            opt_sde.zero_grad(set_to_none=True)
            _, cv = simulate_estimators(model, hedge, ctx, ids, z, "sde")
            loss = ((cv.mean(0) - ctx.target[ids_t]) / ctx.spot).square().mean()
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt_sde.step()
            sde_losses.append(float(loss.detach().cpu())); del z, cv, loss

        model.eval(); hedge.train(); set_requires_grad(model, False); set_requires_grad(hedge, True)
        hedge_losses = []
        for _ in range(ROLLING_STEPS_PER_PHASE):
            ids = sampler.draw()
            z = torch.randn(ROLLING_PATHS, len(ctx.grid_np)-1, dtype=torch.float32, device=DEVICE)
            opt_hedge.zero_grad(set_to_none=True)
            _, cv = simulate_estimators(model, hedge, ctx, ids, z, "hedge")
            loss = torch.var(cv / ctx.spot, dim=0, unbiased=False).mean()
            loss.backward(); nn.utils.clip_grad_norm_(hedge.parameters(), 3.0); opt_hedge.step()
            hedge_losses.append(float(loss.detach().cpu())); del z, cv, loss

        history.append({"update": update, "sde_loss": np.mean(sde_losses), "cv_variance_loss": np.mean(hedge_losses)})
        if update % ROLLING_SAVE_EVERY == 0 or update == ROLLING_UPDATES:
            atomic_save({
                "model_state": cpu_state(model), "hedge_state": cpu_state(hedge),
                "opt_sde": opt_sde.state_dict(), "opt_hedge": opt_hedge.state_dict(),
                "sampler_rng": sampler.rng.bit_generator.state, "torch_rng": torch.get_rng_state(),
                "history": history, "last_update": update,
            }, progress_path)
            pd.DataFrame(history).to_csv(ROLLING_DIR / f"rolling_{date:%Y%m%d}_history.csv", index=False)
        gc.collect()

    model.eval(); set_requires_grad(model, False)
    atomic_save({
        "boundaries": ctx.maturity_boundaries,
        "final_model_state": cpu_state(model),
        "date": str(date.date()),
        "previous_date": str(pd.Timestamp(previous_date).date()),
        "updates": ROLLING_UPDATES,
    }, final_path)
    progress_path.unlink(missing_ok=True)
    del hedge, opt_sde, opt_hedge
    gc.collect()
    return model, ctx



## 6. Development sequence

The first transition uses the rebuilt 1-March forecasting model; 4–7 March use the frozen 15-update rule. This section documents the period used during model development but does not reopen model selection.


In [12]:

stage_log("Development rolling sequence: 1-Mar to 8-Mar")
DEV_DIR = RESULT_DIR / "development_100k"
dev_dates = [pd.Timestamp(x) for x in ["2024-03-01","2024-03-04","2024-03-05","2024-03-06","2024-03-07"]]
dev_next = [pd.Timestamp(x) for x in ["2024-03-04","2024-03-05","2024-03-06","2024-03-07","2024-03-08"]]

# Recreate the fixed 15-update runner's RNG starting point: seed reset, then initial-date model construction.
torch.manual_seed(SEED)
previous = load_maturity_model(INITIAL_DIR / "initial_full_otm_refinement_20240301.pt")
previous_date = INITIAL_DATE

dev_rows = [evaluate_cached(previous, dev_dates[0], dev_next[0], index=1, paths=PRIMARY_EVAL_PATHS, folder=DEV_DIR)]
for local_index, (date, next_date) in enumerate(zip(dev_dates[1:], dev_next[1:]), start=1):
    model, ctx = train_fixed_rolling_day(date, previous, previous_date)
    dev_rows.append(evaluate_cached(model, date, next_date, index=local_index, paths=PRIMARY_EVAL_PATHS, folder=DEV_DIR))
    pd.DataFrame(dev_rows).to_csv(RESULT_DIR / "development_summary_100k.csv", index=False)
    del previous, ctx
    previous, previous_date = model, date
    gc.collect()

development_summary = pd.DataFrame(dev_rows)
display(development_summary[[
    "date","next_date","otm_n","otm_pricing_RMSE","otm_pricing_MAE",
    "otm_NLV_hedge_RMSE","otm_market_delta_hedge_RMSE"
]])
print("Development pooled:", pooled_summary(development_summary))
del previous
gc.collect()


2026-09-25T06:48:58 | Development rolling sequence: 1-Mar to 8-Mar | RSS=596 MiB


Same-day MC:   0%|          | 0/100000 [00:00<?, ?path/s]

2026-09-25T06:49:11 | transition: begin | RSS=583 MiB
2026-09-25T06:49:11 | transition AM=0: simulation begin, rows=6099 | RSS=599 MiB


Future-date MC AM=0:   0%|          | 0/100000 [00:00<?, ?path/s]

2026-09-25T06:49:21 | transition AM=0: simulation returned; terminal storage=10.7 MiB | RSS=606 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.0410958904109589, contracts=155 | RSS=606 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.043835616438356165, contracts=165 | RSS=608 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.04657534246575343, contracts=164 | RSS=608 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.049315068493150684, contracts=203 | RSS=608 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.057534246575342465, contracts=167 | RSS=608 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.06027397260273973, contracts=153 | RSS=608 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.06301369863013699, contracts=171 | RSS=608 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.06575342465753424, contracts=255 | RSS=608 MiB
2026-09-25T06:49:21 | transition AM=0: payoff begin T=0.07671232876712329,

...

Future-date MC AM=1:   0%|          | 0/100000 [00:00<?, ?path/s]

2026-09-25T07:35:17 | transition AM=1: simulation returned; terminal storage=4.2 MiB | RSS=593 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.11506849315068493, contracts=330 | RSS=593 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.1917808219178082, contracts=398 | RSS=594 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.2876712328767123, contracts=415 | RSS=594 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.3643835616438356, contracts=358 | RSS=594 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.4410958904109589, contracts=352 | RSS=594 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.536986301369863, contracts=146 | RSS=594 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.6136986301369863, contracts=119 | RSS=594 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.6904109589041096, contracts=121 | RSS=594 MiB
2026-09-25T07:35:17 | transition AM=1: payoff begin T=0.7863013698630137, contracts=1

,date,next_date,otm_n,otm_pricing_RMSE,otm_pricing_MAE,otm_NLV_hedge_RMSE,otm_market_delta_hedge_RMSE
0,2024-03-01,2024-03-04,3932,1.837789,1.441023,0.663160,0.698004
1,2024-03-04,2024-03-05,3951,4.614017,3.505503,4.084492,1.890195
2,2024-03-05,2024-03-06,4174,5.094898,4.483876,0.509526,1.283119
3,2024-03-06,2024-03-07,3917,4.861550,4.506252,1.513179,3.389083
4,2024-03-07,2024-03-08,4027,3.238063,2.584374,1.547408,0.879009


Development pooled: {'n': 20001, 'pricing_RMSE': 4.126291178975813, 'pricing_MAE': 3.3143494417325456, 'NLV_hedge_RMSE': 2.089672835803063, 'market_delta_hedge_RMSE': 1.8841729942343208}


0


## 7. Untouched out-of-sample test and final 400k evaluation

The test is deliberately restarted as the finalisation run was: PyTorch is reset to the base seed and the saved 7-March checkpoint is reconstructed before 8-March training. This prevents hidden dependence on the in-memory RNG state left by development. Evaluation seeds are indexed 1–14 exactly as in the frozen test run.

After the primary 100k evaluation, the same trained checkpoints are reevaluated with 400k antithetic paths. There is no retraining or model reselection.


In [13]:

stage_log("Frozen out-of-sample test: 8-Mar to 28-Mar")
TEST100_DIR = RESULT_DIR / "test_100k"

# The frozen test was run as a fresh process from the saved 7-Mar checkpoint.
torch.manual_seed(SEED)
previous = load_maturity_model(rolling_checkpoint(pd.Timestamp("2024-03-07")))
previous_date = pd.Timestamp("2024-03-07")

start = trading_dates.index(pd.Timestamp("2024-03-08"))
train_dates = trading_dates[start:-1]
target_dates = trading_dates[start+1:]
assert len(train_dates) == 14 and target_dates[-1] == TEST_END

test_rows = []
for index, (date, next_date) in enumerate(zip(train_dates, target_dates), start=1):
    model, ctx = train_fixed_rolling_day(date, previous, previous_date)
    test_rows.append(evaluate_cached(model, date, next_date, index=index, paths=PRIMARY_EVAL_PATHS, folder=TEST100_DIR))
    pd.DataFrame(test_rows).to_csv(RESULT_DIR / "test_summary_100k.csv", index=False)
    del previous, ctx
    previous, previous_date = model, date
    gc.collect()

test_summary_100k = pd.DataFrame(test_rows)
test_pooled_100k = pooled_summary(test_summary_100k)
print("Frozen test pooled — 100k paths")
print(pd.Series(test_pooled_100k).to_string())
display(test_summary_100k[[
    "date","next_date","otm_n","otm_pricing_RMSE",
    "otm_NLV_hedge_RMSE","otm_market_delta_hedge_RMSE"
]])
del previous
gc.collect()


2026-09-25T07:35:18 | Frozen out-of-sample test: 8-Mar to 28-Mar | RSS=576 MiB
Warm-start 2024-03-07 -> 2024-03-08: 36 heads | elapsed=0.00274y | median head mismatch=0.00000y


rolling 2024-03-08:   0%|          | 0/15 [00:00<?, ?SDE/s]

Same-day MC:   0%|          | 0/100000 [00:00<?, ?path/s]

2026-09-25T07:46:15 | transition: begin | RSS=639 MiB
2026-09-25T07:46:15 | transition AM=0: simulation begin, rows=6554 | RSS=652 MiB


Future-date MC AM=0:   0%|          | 0/100000 [00:00<?, ?path/s]

...

2026-09-25T10:15:32 | transition AM=0: simulation returned; terminal storage=10.7 MiB | RSS=687 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.0410958904109589, contracts=169 | RSS=687 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.049315068493150684, contracts=164 | RSS=689 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.052054794520547946, contracts=161 | RSS=689 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.0547945205479452, contracts=168 | RSS=689 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.057534246575342465, contracts=158 | RSS=689 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.06027397260273973, contracts=231 | RSS=689 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.0684931506849315, contracts=148 | RSS=689 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.07123287671232877, contracts=145 | RSS=690 MiB
2026-09-25T10:15:32 | transition AM=0: payoff begin T=0.07397260273972603, c

Future-date MC AM=1:   0%|          | 0/100000 [00:00<?, ?path/s]

2026-09-25T10:15:43 | transition AM=1: simulation returned; terminal storage=4.6 MiB | RSS=683 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.06027397260273973, contracts=235 | RSS=683 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.136986301369863, contracts=370 | RSS=684 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.2328767123287671, contracts=429 | RSS=685 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.3095890410958904, contracts=447 | RSS=685 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.3863013698630137, contracts=366 | RSS=685 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.4821917808219178, contracts=360 | RSS=685 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.5589041095890411, contracts=152 | RSS=685 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.6356164383561644, contracts=156 | RSS=685 MiB
2026-09-25T10:15:43 | transition AM=1: payoff begin T=0.7315068493150685, contracts=1

,date,next_date,otm_n,otm_pricing_RMSE,otm_NLV_hedge_RMSE,otm_market_delta_hedge_RMSE
0,2024-03-08,2024-03-11,4206,2.299408,0.980097,1.220444
1,2024-03-11,2024-03-12,3780,5.037866,3.482207,0.557066
2,2024-03-12,2024-03-13,3943,3.054121,1.140808,1.552858
3,2024-03-13,2024-03-14,3984,2.475281,2.167849,1.500277
4,2024-03-14,2024-03-15,4283,3.226200,1.795468,0.672911
5,2024-03-15,2024-03-18,4156,2.577106,2.366401,0.716774
6,2024-03-18,2024-03-19,4207,2.644334,2.780860,1.352368
7,2024-03-19,2024-03-20,3957,4.318681,2.080231,1.479892
8,2024-03-20,2024-03-21,4018,3.467688,1.608245,1.185172
9,2024-03-21,2024-03-22,4200,1.991789,0.922530,1.191907


0

In [14]:

stage_log("400k-path evaluation of the frozen test checkpoints")
TEST400_DIR = RESULT_DIR / "test_400k"
rows400 = []
for index, (date, next_date) in enumerate(zip(train_dates, target_dates), start=1):
    model = load_maturity_model(rolling_checkpoint(date))
    rows400.append(evaluate_cached(model, date, next_date, index=index, paths=FINAL_EVAL_PATHS, folder=TEST400_DIR))
    pd.DataFrame(rows400).to_csv(RESULT_DIR / "test_summary_400k.csv", index=False)
    del model
    gc.collect()

test_summary_400k = pd.DataFrame(rows400)
test_pooled_400k = pooled_summary(test_summary_400k)
print("Frozen test pooled — 400k paths")
print(pd.Series(test_pooled_400k).to_string())

pd.DataFrame([
    {"paths": PRIMARY_EVAL_PATHS, **test_pooled_100k},
    {"paths": FINAL_EVAL_PATHS, **test_pooled_400k},
]).to_csv(RESULT_DIR / "test_100k_vs_400k.csv", index=False)



# Contract-level diagnostics are descriptive only; they do not reopen model selection.
frames = []
for date in train_dates:
    p = TEST400_DIR / f"{pd.Timestamp(date):%Y%m%d}_contracts.csv"
    x = pd.read_csv(p)
    mask = x["otm_on_both_dates"]
    if mask.dtype != bool: mask = mask.astype(str).str.lower().eq("true")
    frames.append(x.loc[mask].copy())
test_contracts = pd.concat(frames, ignore_index=True)

test_contracts["days_next"] = test_contracts["T_next"] * 365.0
test_contracts["maturity_bucket"] = pd.cut(
    test_contracts["days_next"], [0,30,90,180,366],
    labels=["7-30d","31-90d","91-180d","181-365d"], include_lowest=True
)
test_contracts["moneyness_bucket"] = pd.cut(
    test_contracts["k_next"], [-np.inf,-.10,-.05,0,.05,.10,np.inf],
    labels=["k<-10%","-10:-5%","-5:0%","0:5%","5:10%","k>10%"]
)

def error_table(group):
    rows = []
    for key, g in test_contracts.groupby(group, observed=True):
        pe = g["next_day_pricing_error"].to_numpy(float)
        he = g["hedge_error"].dropna().to_numpy(float)
        me = g["market_delta_hedge_error"].dropna().to_numpy(float)
        rows.append({
            group: key, "n": len(g),
            "pricing_RMSE": np.sqrt(np.mean(pe**2)), "pricing_MAE": np.mean(np.abs(pe)),
            "NLV_hedge_RMSE": np.sqrt(np.mean(he**2)),
            "market_delta_hedge_RMSE": np.sqrt(np.mean(me**2)),
        })
    return pd.DataFrame(rows)

for group in ("cp_flag","maturity_bucket","moneyness_bucket"):
    table = error_table(group)
    display(table)
    table.to_csv(RESULT_DIR / f"test_400k_by_{group}.csv", index=False)

delta_pairs = test_contracts[["neural_lv_delta","delta_market"]].dropna()
delta_corr = pearson_reduction(delta_pairs["neural_lv_delta"], delta_pairs["delta_market"])
print("Pooled NLV / OptionMetrics delta correlation:", f"{delta_corr:.6f}")



plot_df = test_summary_400k.copy()
plot_df["next_date"] = pd.to_datetime(plot_df["next_date"])

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(plot_df["next_date"], plot_df["otm_pricing_RMSE"], marker="o")
ax.set(title="Neural LV: next-day OTM pricing RMSE", xlabel="Target date", ylabel="RMSE")
ax.grid(alpha=.25); fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(PLOT_DIR / "test_pricing_rmse.png", dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(plot_df["next_date"], plot_df["otm_NLV_hedge_RMSE"], marker="o", label="Neural-LV delta")
ax.plot(plot_df["next_date"], plot_df["otm_market_delta_hedge_RMSE"], marker="o", label="OptionMetrics delta")
ax.set(title="One-day hedge RMSE", xlabel="Target date", ylabel="RMSE")
ax.legend(); ax.grid(alpha=.25); fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(PLOT_DIR / "test_hedge_rmse.png", dpi=160); plt.show()

final_manifest = {
    "model": "Neural Local Volatility — final CPU baseline",
    "initial_calibration": [
        "v5 shared 4x50, 1000 alternating epochs, capped OTM, maturity-balanced sampling",
        "Stage II maturity-head 4x50, 600 alternating epochs, deterministic 80% full-OTM train split, uniform 400-contract sampling, raw MSE, constant Adam 1e-4/1e-3",
        "v7 crash-safe full-OTM refinement, max 30 updates, min 10, patience 12, update-0 guaranteed in 50k rerank",
    ],
    "rolling_recalibration": "fixed 15 updates/day; all filtered OTM; 4096 paths; 400 contracts; 10 NLV + 10 CV steps/update; Adam 1e-4/1e-3; deterministic pre-construction CV seed; no early stopping or reranking",
    "evaluation": "100k common-random-number primary; 400k antithetic final test; self-financing one-day hedge error with financing/dividend carry; 2k path chunks; sequential AM/PM groups",
    "development": "2024-03-01 to 2024-03-08",
    "test": "2024-03-08 to 2024-03-28",
    "test_100k": test_pooled_100k,
    "test_400k": test_pooled_400k,
    "delta_correlation_400k": float(delta_corr),
}
(CACHE_DIR / "final_manifest.json").write_text(json.dumps(final_manifest, indent=2, default=str), encoding="utf-8")
print(json.dumps(final_manifest, indent=2, default=str))
stage_log("Notebook complete")


2026-09-25T10:15:45 | 400k-path evaluation of the frozen test checkpoints | RSS=665 MiB


Same-day MC:   0%|          | 0/400000 [00:00<?, ?path/s]

2026-09-25T10:16:38 | transition: begin | RSS=659 MiB
2026-09-25T10:16:38 | transition AM=0: simulation begin, rows=6554 | RSS=667 MiB


Future-date MC AM=0:   0%|          | 0/400000 [00:00<?, ?path/s]

2026-09-25T10:17:19 | transition AM=0: simulation returned; terminal storage=42.7 MiB | RSS=710 MiB
2026-09-25T10:17:19 | transition AM=0: payoff begin T=0.0410958904109589, contracts=191 | RSS=710 MiB
2026-09-25T10:17:19 | transition AM=0: payoff begin T=0.043835616438356165, contracts=191 | RSS=710 MiB
2026-09-25T10:17:20 | transition AM=0: payoff begin T=0.04657534246575343, contracts=248 | RSS=710 MiB
2026-09-25T10:17:20 | transition AM=0: payoff begin T=0.057534246575342465, contracts=188 | RSS=710 MiB
2026-09-25T10:17:20 | transition AM=0: payoff begin T=0.06027397260273973, contracts=173 | RSS=710 MiB
2026-09-25T10:17:20 | transition AM=0: payoff begin T=0.06301369863013699, contracts=175 | RSS=710 MiB
2026-09-25T10:17:20 | transition AM=0: payoff begin T=0.06575342465753424, contracts=152 | RSS=710 MiB
2026-09-25T10:17:20 | transition AM=0: payoff begin T=0.0684931506849315, contracts=249 | RSS=710 MiB
2026-09-25T10:17:20 | transition AM=0: payoff begin T=0.07671232876712329, c

...

,maturity_bucket,n,pricing_RMSE,pricing_MAE,NLV_hedge_RMSE,market_delta_hedge_RMSE
0,7-30d,11669,2.178839,1.809350,1.790565,1.509614
1,31-90d,17289,2.525337,2.041963,1.819593,1.277288
2,91-180d,20258,3.470440,2.796424,1.935679,1.181748
3,181-365d,8406,4.645939,3.753637,2.037315,1.289978


,moneyness_bucket,n,pricing_RMSE,pricing_MAE,NLV_hedge_RMSE,market_delta_hedge_RMSE
0,k<-10%,12644,3.751736,3.082386,1.604300,0.750992
1,-10:-5%,11645,3.476330,2.879917,2.047012,1.063372
2,-5:0%,16843,3.038234,2.407414,2.252522,1.653498
3,0:5%,12348,2.900183,2.125793,1.671796,1.501506
4,5:10%,3466,1.588201,1.122875,1.040446,0.876300
5,k>10%,676,2.523573,2.100983,0.757603,0.582393


Pooled NLV / OptionMetrics delta correlation: 0.996703


: 

: 